# Contract API v2.0 (systems ↔ systems)

Ниже описан контрактное API для взаимодействия систем через брокеры (Kafka/Mosquitto) и центральный журнал событий.

Ключевое требование (Запрос 6): при выборе контрагента **обязательно учитываются `security_goals`**. Описания целей хранятся у Регулятора; в сообщениях передаются только их **ID**.

## 1) Envelope (request/response)

Базовый envelope использует поле `action` для маршрутизации.

Минимальные поля request (envelope):
- `action: str` — действие/тип обработки (используется как ключ маршрутизации)
- `payload: dict` — данные сообщения (сценарные поля/параметры)
- `sender: str` — идентификатор отправителя (система/компонент)
- `sender_role: str` — роль отправителя (например, `aggregator`, `system`, `admin` для owner/компонента)
- `timestamp: <number|iso>` — временная метка отправки
- `security_internal: bool` — признак «внутреннего» сообщения (для цепочек с security-monitor)

Request/response поверх брокера добавляет (со стороны transport/SystemBus):
- `correlation_id: str` — связывает request и response
- `reply_to: str` — топик/канал, куда отправляется ответ

Shape response (в проекте используется `create_response()`):
- `action: "response"`
- `payload: dict`
- `sender: str`
- `correlation_id: str`
- `success: bool`
- `error?: str`

---
## 2) Центральный журнал (EventJournal)

Центральный журнал **принимает сообщения через брокер** и переиспользует текущий контракт EventJournal. Он записывает как внешние события между системами, так и внутренние события, поступающие из внутренних журналов компонентов.

Базовый контракт входящего сообщения в журнал (`EventJournal`):
- `action: "emit_event"`
- `payload` содержит:
  - `event_type: str`
  - `severity: str` (`info|warning|error|security|audit`)
  - `source_component: str` — откуда пришло событие
  - `payload: dict` — доменные атрибуты события
  - `trace_context?: dict` — опционально

Минимальный response журнала (ack):
- `status: "accepted"`
- `event_type: str`
- `severity: str`

---
## 3) `security_goals` (v2.0)

### 3.1 Представление
- Передаём `security_goals: List[str]` — **ID** целей безопасности.
- Для заказов различают `scenario_security_goals` (задаёт Заказчик явно) и `order_security_goals` (формирует агрегатор).
- ID имеют вид, как минимум, из текущего реестра (пример): `ЦБ-OPS-1`, `SG-FM-001`, `SG-SM-001`, `SG-MP-001`, `SG-BL-001`.

### 3.2 Где цели берутся
- Для каждой системы: `system_security_goals: List[str]` определяется при проектировании по реестру целей (у Регулятора).
- Для каждого заказа: `order_security_goals: List[str]` **агрегатор формирует** из `scenario_security_goals` (заданы Заказчиком в запросе) и контекста заказа (тип задачи и координаты) по правилам v2.0.

### 3.3 Два момента фильтрации (оба обязательны)
1) Агрегатор (между receive_order и формированием предложения/выбором исполнителя):
   - отвергает контрагента, если его `system_security_goals` **не покрывают** `order_security_goals` (правило покрытия определяется v2.0)
2) ОрВД БАС (между UTM-approval и validate mission):
   - валидирует, что миссия и исполнитель соответствуют требованиям целей
+

Для отклонения рекомендуется стандартный блок в ответах/ошибках:
- `rejection: { rejected_by_security_goals: List[str], reason_code: str, details?: dict }`

### 3.4 Расширяемость (delivery/inspection)
Для других сценариев вводится общее поле:
- `scenario: { type: "delivery"|"inspection"|..., details: dict }`

и финальный набор `order_security_goals` агрегатор формирует детерминированно как функцию от `scenario_security_goals` и контекста заказа (тип/координаты).


## 4) Состав сообщений между системами (пока — только поля/контракты обработки)

Ниже: минимальный набор полей request/response и рекомендации по обработке `security_goals`.

### 4.1 Customer → Aggregator
- Action: `receive_order`
- Request payload (минимально):
  - `order: { id, pickup, dropoff, payload_weight, distance_km, payload_value, ... }`
  - `scenario_security_goals: List[str]` — цб сценария, задаются Заказчиком явно
  - (опционально) `scenario`
- Response payload:
  - `status`
  - `order_security_goals: List[str]` — целевой набор для фильтрации контрагента (формируется агрегатором)
  - (опционально) `security_goals_selection_mode: "derived_minimum"|"any_by_cost"`

### 4.2 Aggregator → Operator
- Action: `receive_order`
- Request payload:
  - `order: OrderV2`
  - `order_security_goals: List[str]` — сформирован агрегатором по правилам v2.0 (может быть `[]` в режиме `any_by_cost`)
  - (опционально) `applied_security_goals` для диагностики
- Response payload (успех):
  - `proposal: { proposal_id, price, margin_percent, delivery_time?, applied_security_goals }`
- Response payload (ошибка/отбраковка):
  - `error: str`
  - `rejection?: { rejected_by_security_goals, reason_code, details? }`

### 4.3 Operator → Insurer
- Action: `request_insurance_quote`
- Request payload:
  - `mission_details: { ... }`
  - `order_security_goals: List[str]` (для привязки полиса к требованиям)
- Response payload:
  - `quote_id`
  - `premium`
  - `coverage`
  - `valid_until`

### 4.4 Operator → Developers (БАС)
- Action: `purchase_uas` (или `find_available_uas` для этапа выбора)
- Request payload (purchase):
  - `developer_id`
  - `model_id`
  - `quantity`
  - `order_security_goals: List[str]` (если требуется квалификация/сертификация)
- Response payload:
  - `success: bool`
  - `order_id`/`delivery`-поля (как реализовано у DeveloperClient)

### 4.5 Operator ↔ DronePortGCS
- Action (пример): `gcs_mission_requested` / `gcs_mission_provided`
- Request payload:
  - `mission_draft`
  - `order_security_goals: List[str]`
- Response payload:
  - `gcs_mission_config`
  - `applied_security_goals`

### 4.6 Operator ↔ NUS
- Action (пример): `request_nus_mission` / `nus_mission_details`
- Request payload:
  - `mission_requirements`
  - `order_security_goals: List[str]`
- Response payload:
  - `mission_details`
  - `validation_security_goals?: List[str]`

### 4.7 NUS ↔ Agro-drone
- Action (пример): `agro_order_received` / `agro_uas_selected`
- Request payload:
  - `agro_task`
  - `order_security_goals: List[str]`
- Response payload:
  - `selected_uas`
  - `agro_mission_plan`

### 4.8 Agro-drone ↔ SITL
- Action: `sitl_simulate`
- Request payload:
  - `mission_id`
  - `scenario`
- Response payload:
  - `simulation_result`

### 4.9 Operator ↔ UTM/OrВД БАС
- Action (пример): `request_utm_approval` / `validate_mission`
- Request payload:
  - `mission_id`
  - `mission_security_goals: List[str]` (обычно = order_security_goals + scenario_goals_addon)
- Response payload:
  - `approved: bool`
  - `approval_id`
  - (если not approved) `rejection` по `mission_security_goals`

### 4.10 Всеми системами → Regulator
- Action: `get_system_topics`, `get_security_goals`, `get_regulations`, `report_incident` (v2.0)
- Request payload (пример для get_security_goals):
  - `system_type` или `component_name`
- Response payload:
  - `security_goals: [{ goal_id, description?, mappings? }]`

---
## 5) Правила обработки полей (минимальные)
- `security_goals` считаются *детерминированным* набором ID.
- Если `order_security_goals=[]`, агрегатор пытается определить минимальный набор цб для конкретной задачи; если набор не найден, фильтрация допускает любых контрагентов, а окончательный выбор делается Заказчиком по стоимости.
- При отбраковке `rejection.rejected_by_security_goals` содержит **только те goal_id**, которые послужили причиной.
- Любые сценарные поля `scenario.details` расширяемы без изменения envelope.


In [ ]:
# Пример: pydantic-модели контрактов v2.0 + заполнение ключевых сообщений.
# Notebook ориентирован на описание контракта (не на запуск реального брокера).

from typing import Any, Dict, List, Optional, Literal

try:
    from pydantic import BaseModel, Field
    PYDANTIC_AVAILABLE = True
except Exception:  # noqa: BLE001
    PYDANTIC_AVAILABLE = False

    # Минимальный fallback, чтобы примерные instantiation'ы работали
    # даже без pydantic в окружении.
    class BaseModel:  # type: ignore
        def __init__(self, **data: Any):
            for k, v in data.items():
                setattr(self, k, v)

        def model_dump(self) -> Dict[str, Any]:
            return dict(self.__dict__)

    def Field(*_args: Any, **kwargs: Any) -> Any:  # type: ignore
        # pydantic Field обычно задаёт metadata и defaults; в примере достаточно вернуть default.
        return kwargs.get("default")


class RejectionV2(BaseModel):
    rejected_by_security_goals: List[str]
    reason_code: str
    details: Optional[Dict[str, Any]] = None


class ScenarioV2(BaseModel):
    type: Literal['delivery', 'inspection', 'order', 'agro', 'unknown']
    details: Dict[str, Any] = {}


class OrderV2(BaseModel):
    id: str
    pickup: Dict[str, float]
    dropoff: Dict[str, float]
    payload_weight: float
    distance_km: float
    payload_value: float
    scenario: Optional[ScenarioV2] = None
    # ЦБ задаются Заказчиком явно; агрегатор формирует финальный набор для фильтрации
    scenario_security_goals: List[str]
    # Финальный набор для фильтрации контрагента (агрегатор формирует)
    # Если набор не определён, допускается режим выбора "any_by_cost"
    order_security_goals: Optional[List[str]] = None


class ProposalV2(BaseModel):
    proposal_id: str
    price: float
    margin_percent: float
    delivery_time: Optional[str] = None
    applied_security_goals: List[str]


class OperatorReceiveOrderResponsePayloadV2(BaseModel):
    proposal: Optional[ProposalV2] = None
    # ошибка или отбраковка
    error: Optional[str] = None
    rejection: Optional[RejectionV2] = None


class EnvelopeV2(BaseModel):
    action: str
    payload: Dict[str, Any]
    sender: str
    sender_role: str
    timestamp: float
    security_internal: bool = False
    # transport добавит это при request/response
    correlation_id: Optional[str] = None
    reply_to: Optional[str] = None


# ------------------------------
# Пример данных `security_goals`
# ------------------------------

# ЦБ (scenario_security_goals) сценария задаются Заказчиком явно в запросе
scenario_security_goals = [
    'ЦБ-OPS-1',  # авторизация команд
    'ЦБ-OPS-4',  # экономическая эффективность
]

# Система/компонент (например, бизнес-логика оператора) поддерживает свои goal_id
operator_system_security_goals = {
    'operator-business-logic': ['SG-BL-001', 'ЦБ-OPS-4'],
    'operator-fleet-manager': ['SG-FM-001', 'ЦБ-OPS-1', 'ЦБ-OPS-5'],
}

# Customer -> Aggregator: Заказчик задаёт scenario_security_goals явно
order_customer = OrderV2(
    id='ORDER-EXAMPLE-001',
    pickup={'lat': 55.76, 'lon': 37.62},
    dropoff={'lat': 55.75, 'lon': 37.61},
    payload_weight=1.0,
    distance_km=2.0,
    payload_value=20000,
    scenario=ScenarioV2(type='order', details={'purpose': 'cargo'}),
    scenario_security_goals=scenario_security_goals,
    order_security_goals=[],
)

# Aggregator -> Operator: агрегатор формирует order_security_goals (пример)
order_security_goals_derived = list(scenario_security_goals)
order_for_operator = OrderV2(
    id=order_customer.id,
    pickup=order_customer.pickup,
    dropoff=order_customer.dropoff,
    payload_weight=order_customer.payload_weight,
    distance_km=order_customer.distance_km,
    payload_value=order_customer.payload_value,
    scenario=order_customer.scenario,
    scenario_security_goals=scenario_security_goals,
    order_security_goals=order_security_goals_derived,
)

# Пример ответа оператора (успех): proposal + применённые goals
proposal = ProposalV2(
    proposal_id='PROP-EXAMPLE-123',
    price=105000.0,
    margin_percent=12.5,
    delivery_time='2026-03-30T10:00:00Z',
    applied_security_goals=['ЦБ-OPS-1', 'ЦБ-OPS-4'],
)

operator_receive_order_response = OperatorReceiveOrderResponsePayloadV2(
    proposal=proposal,
)

# Пример ответа оператора (отбраковка): rejected_by_security_goals
operator_rejection_response = OperatorReceiveOrderResponsePayloadV2(
    error='No suitable operator candidate after security filtering',
    rejection=RejectionV2(
        rejected_by_security_goals=['ЦБ-OPS-4'],
        reason_code='goal_coverage_mismatch',
        details={'candidate': 'operator-002'},
    ),
)

# ------------------------------
# Пример envelope для receive_order (Customer -> Aggregator)
# ------------------------------

receive_order_customer = EnvelopeV2(
    action='receive_order',
    payload=order_customer.model_dump() if hasattr(order_customer, 'model_dump') else order_customer.__dict__,
    sender='customer-001',
    sender_role='customer',
    timestamp=0.0,
    security_internal=False,
)

# ------------------------------
# Пример central journal emit_event
# ------------------------------

event_journal_emit = {
    'action': 'emit_event',
    'payload': {
        'event_type': 'security_goals_rejection',
        'severity': 'security',
        'source_component': 'operator_system',
        'payload': {
            'order_id': order_for_operator.id,
            'rejection': operator_rejection_response.rejection.model_dump()
            if operator_rejection_response.rejection and hasattr(operator_rejection_response.rejection, 'model_dump')
            else None,
        },
        # trace_context: optional
    },
}

print('Example order_security_goals:', order_for_operator.order_security_goals)
print('Example proposal_id:', proposal.proposal_id)
print('Example journal emit payload.event_type:', event_journal_emit['payload']['event_type'])


## 6) Предлагаемые `event_type` для реализации бизнес-логики v2.0

Ниже — логичные типы событий для центрального `EventJournal`, которые покрывают весь поток: получение заказа, формирование/фильтрацию `security_goals`, подбор исполнителя, страхование, миссию (включая OrВД/UTM) и этапы `delivery/inspection`.

### `security_goals` (формирование, фильтрация, rejection)
- `security_goals_received` — агрегатор получил `scenario_security_goals` от Заказчика
- `order_security_goals_formed` — агрегатор сформировал финальный `order_security_goals` (derived_minimum/any_by_cost)
- `security_goals_filtering_started` — старт фильтрации контрагентов по coverage
- `security_goals_contractor_rejected` — контрагент отклонён из-за неподдержанных целей
- `security_goals_rejection` — зафиксированы `rejected_by_security_goals` и `reason_code`
- `security_goals_mission_requirements_built` — ОрВД подготовил требования для validate mission
- `security_goals_mission_approved` — миссия одобрена по требованиям goals
- `security_goals_mission_denied` — миссия отклонена по требованиям goals

### Жизненный цикл заказа (Customer ↔ Aggregator ↔ Operator)
- `aggregator_order_received` — агрегатор принял order от Заказчика
- `proposal_requested` — агрегатор запросил предложения у операторов/исполнителей
- `proposal_calculated_by_operator` — оператор посчитал предложение
- `proposal_submitted_to_aggregator` — оператор передал proposal агрегатору
- `customer_selected_contractor` — заказчик выбрал исполнителя по предложениям/стоимости
- `operator_assigned_to_order` — оператор назначен на выполнение заказа
- `order_execution_started` — оператор стартовал выполнение
- `order_execution_completed` — заказ успешно завершён
- `order_execution_rejected` — заказ отклонён на бизнес-логическом уровне

### Страхование / политика
(можно переиспользовать уже существующие в проекте `insurer_quote_*` event_type)
- `policy_cost_approved` — страховщик одобрил стоимость/условия полиса
- `policy_cost_denied` — страховщик отказал в покрытии/стоимости

### Миссия / OrВД/UTM / DronePort / полёт
- `mission_requirements_built` — требования миссии сформированы из заказа и целей
- `mission_planned` — миссия построена планировщиком
- `utm_approval_requested` — запрос на approval отправлен в UTM/OrВД
- `utm_approval_approved` — approval получен
- `utm_approval_denied` — approval отклонён
- `droneport_entry_authorized` — доступ в DronePort для старта разрешён
- `droneport_entry_denied` — доступ в DronePort для старта запрещён
- `mission_started` — миссия запущена
- `mission_completed` — миссия завершена

### `delivery/inspection` (для сценариев расширения)
- `delivery_started` — доставка начата
- `delivery_completed` — доставка завершена
- `inspection_started` — инспекция начата
- `inspection_completed` — инспекция завершена
- `inspection_failed` — инспекция неуспешна

### Регулятор и инциденты
- `incident_report_submitted` — инцидент подан на регулятора
- `incident_recorded` — инцидент зафиксирован в системе регулятора
